#  TRABAJO FINAL LP2
#### PARTE 1
Hecha por:
-  Pariona Huarhauchi Misael Gabriel - 20240725

In [1]:
pip install selenium pandas webdriver-manager

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install --upgrade webdriver-manager selenium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import NoSuchElementException, TimeoutException

def configurar_driver():
    """Configura el navegador simulando ser un usuario humano."""
    opciones = Options()
    opciones.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36")
    opciones.add_argument("--disable-blink-features=AutomationControlled")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    
    servicio = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=servicio, options=opciones)
    driver.set_page_load_timeout(20) 
    return driver

def obtener_enlaces_ebay(driver, cantidad_objetivo=50):
    """FASE 1: Extrae los enlaces específicos del iPhone 17 Pro Max."""
    url_busqueda = "https://www.ebay.com/sch/i.html?_nkw=iphone+17+pro+max"
    print(f"Abriendo la página de listado: {url_busqueda}")
    
    try:
        driver.get(url_busqueda)
    except TimeoutException:
        driver.execute_script("window.stop();")
    
    # Pausa manual para evadir el Firewall (¡Como te funcionó la última vez!)
    print("\n" + "="*60)
    print("PAUSA DE SEGURIDAD ACTIVADA")
    print("1. Ve a la ventana de Chrome que se abrió.")
    print("2. Si eBay te pide verificar que eres humano, resuélvelo.")
    print("3. Asegúrate de estar viendo los iPhone 17 Pro Max en pantalla.")
    print("4. Vuelve a esta consola negra...")
    input("PRESIONA LA TECLA 'ENTER' AQUÍ PARA QUE EL BOT ARRANQUE: ")
    print("="*60 + "\n")
    
    enlaces = set() 
    
    while len(enlaces) < cantidad_objetivo:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 2);")
        time.sleep(2)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        
        elementos_a = driver.find_elements(By.TAG_NAME, "a")
        
        for a in elementos_a:
            href = a.get_attribute("href")
            if href and "/itm/" in href:
                link_limpio = href.split("?")[0]
                enlaces.add(link_limpio)
                if len(enlaces) >= cantidad_objetivo:
                    break
        
        print(f"-> Se han recolectado {len(enlaces)} enlaces de iPhone 17 Pro Max ({cantidad_objetivo} objetivo)...")
        
        if len(enlaces) < cantidad_objetivo:
            try:
                boton_siguiente = driver.find_element(By.CSS_SELECTOR, "a.pagination__next")
                try:
                    driver.get(boton_siguiente.get_attribute("href"))
                except TimeoutException:
                    driver.execute_script("window.stop();")
                time.sleep(random.uniform(3, 5))
            except NoSuchElementException:
                print("No hay más páginas disponibles.")
                break

    return list(enlaces)[:cantidad_objetivo]

def extraer_detalle_ebay(driver, url, indice, total):
    """FASE 2: Ingresa a la página de detalle del iPhone y extrae las variables."""
    print(f"[{indice}/{total}] Extrayendo datos de: {url}")
    
    try:
        driver.get(url)
    except TimeoutException:
        print("   [!] Timeout en esta página, forzando lectura...")
        driver.execute_script("window.stop();")
    
    time.sleep(random.uniform(2, 4))
    
    datos = {
        "URL": url,
        "Titulo": "No encontrado",
        "Precio": "No encontrado",
        "Condicion": "No encontrada",
        "Vendedor": "No encontrado"
    }
    
    try:
        titulo_elem = driver.find_element(By.CSS_SELECTOR, "h1.x-item-title__mainTitle")
        datos["Titulo"] = titulo_elem.text
    except NoSuchElementException: pass

    try:
        precio_elem = driver.find_element(By.CSS_SELECTOR, "div.x-price-primary span.ux-textspans")
        datos["Precio"] = precio_elem.text
    except NoSuchElementException: pass

    try:
        condicion_elem = driver.find_element(By.CSS_SELECTOR, "div.x-item-condition-value span.ux-textspans")
        datos["Condicion"] = condicion_elem.text
    except NoSuchElementException: pass

    try:
        vendedor_elem = driver.find_element(By.CSS_SELECTOR, "div.x-sellercard-atf__info__about-seller span.ux-textspans")
        datos["Vendedor"] = vendedor_elem.text
    except NoSuchElementException: pass

    return datos

def main():
    driver = configurar_driver()
    datos_extraidos = []
    cantidad_a_extraer = 50
    
    try:
        print("--- INICIANDO FASE 1: RECOPILACIÓN DE LINKS (iPHONE 17 PRO MAX) ---")
        enlaces_productos = obtener_enlaces_ebay(driver, cantidad_objetivo=cantidad_a_extraer)
        
        if len(enlaces_productos) == 0:
            print("\n CRÍTICO: El bot no vio los enlaces. Asegúrate de presionar ENTER solo cuando cargue la página.")
            return 
            
        print(f"\n¡Se encontraron {len(enlaces_productos)} iPhones! Iniciando extracción...\n")
        
        print("--- INICIANDO FASE 2: EXTRACCIÓN DE DETALLES ---")
        for indice, url in enumerate(enlaces_productos, start=1):
            resultado = extraer_detalle_ebay(driver, url, indice, len(enlaces_productos))
            if resultado:
                datos_extraidos.append(resultado)
                
    except Exception as e:
        print(f"\nOcurrió un error inesperado: {e}")
                
    finally:
        driver.quit()
        print("\nNavegador cerrado de forma segura.")

    if datos_extraidos:
        df = pd.DataFrame(datos_extraidos)
        # Nombre del archivo de salida
        nombre_archivo = "dataset_iphone17_ebay.csv"
        df.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
        print(f"\n¡TRABAJO TERMINADO! {len(datos_extraidos)} registros guardados exitosamente en '{nombre_archivo}'.")

if __name__ == "__main__":
    main()

--- INICIANDO FASE 1: RECOPILACIÓN DE LINKS (iPHONE 17 PRO MAX) ---
Abriendo la página de listado: https://www.ebay.com/sch/i.html?_nkw=iphone+17+pro+max

PAUSA DE SEGURIDAD ACTIVADA
1. Ve a la ventana de Chrome que se abrió.
2. Si eBay te pide verificar que eres humano, resuélvelo.
3. Asegúrate de estar viendo los iPhone 17 Pro Max en pantalla.
4. Vuelve a esta consola negra...

-> Se han recolectado 50 enlaces de iPhone 17 Pro Max (50 objetivo)...

¡Se encontraron 50 iPhones! Iniciando extracción...

--- INICIANDO FASE 2: EXTRACCIÓN DE DETALLES ---
[1/50] Extrayendo datos de: https://www.ebay.com/itm/226423631540
[2/50] Extrayendo datos de: https://www.ebay.com/itm/285596667873
[3/50] Extrayendo datos de: https://www.ebay.com/itm/318494439702
[4/50] Extrayendo datos de: https://www.ebay.com/itm/176588460140
[5/50] Extrayendo datos de: https://www.ebay.com/itm/366454754961
[6/50] Extrayendo datos de: https://www.ebay.com/itm/146563753838
[7/50] Extrayendo datos de: https://www.ebay.co